In [1]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier, VotingClassifier

# Loading data

In [2]:
learn_data = pd.read_csv("minimal_train_fs.csv", header = None)
learn_data.columns = ['Age', 'TB', 'Alkphos', 'Sgot', 'ALB', 'AR', 'BilRatio', 'Female', 'Target']
learn_data["Female"] = learn_data["Female"].astype("category")
learn_data["Target"] = learn_data["Target"].astype("category")
learn_data.head()

,Age,TB,Alkphos,Sgot,ALB,AR,BilRatio,Female,Target
0,48,1.504077,5.641907,4.304065,2.4,0.52,0.511111,0,0
1,39,0.641854,5.192957,4.127134,4.3,1.38,0.473684,0,0
2,23,0.000000,5.356586,4.382027,3.1,1.00,0.300000,0,0
3,42,-0.356675,5.023881,4.394449,3.2,1.06,0.285714,1,0
4,54,3.117950,6.324359,3.610918,3.4,0.80,0.504425,1,0


In [3]:
learn_data.isna().value_counts()

Age    TB     Alkphos  Sgot   ALB    AR     BilRatio  Female  Target
False  False  False    False  False  False  False     False   False     449
Name: count, dtype: int64

In [4]:
X = learn_data.drop(columns = ["Target"])
y = learn_data["Target"]

# Metrics

In [5]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
import numpy as np

def compute_metrics (y_real, y_pred) -> list[float]:
    F1_macro = f1_score(y_real, y_pred, average = "macro")
    recall = recall_score(y_real, y_pred, average = "macro")
    prec = precision_score(y_real, y_pred, average = "macro")
    acc = accuracy_score(y_real, y_pred)
    return [F1_macro, recall, prec, acc]

def confusion (y_real, y_pred) -> None:
    TP = sum(np.logical_and(y_real == y_pred, y_real == 1))
    TN = sum(np.logical_and(y_real == y_pred, y_real == 0))
    FP = sum(np.logical_and(y_real != y_pred, y_real == 0))
    FN = sum(np.logical_and(y_real != y_pred, y_real == 1))
    print("\t\tPredicted")
    print("\t\t+1\t0")
    print(f"Real\t+1\t{TP}\t{FN}")
    print(f"\t0\t{FP}\t{TN}")
    print(f"Accuracy: {((TP + TN) / y_real.shape[0] * 100):.2f}%".format())

metrics_df = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

# The classifiers, by themselves
## QDA

In [6]:
QDA_pipeline = Pipeline([('scaler', StandardScaler()), ('QDA', QuadraticDiscriminantAnalysis(priors = (0.5, 0.5)))])

n = 10
regs = np.logspace(start = -4, stop = -0.5, num = 100)

QDA_search = GridSearchCV(estimator = QDA_pipeline,
                          param_grid = {'QDA__reg_param' : regs},
                          scoring = 'f1_macro',
                          cv = 5)
QDA_search.fit(X, y)
QDA_search.best_params_

{'QDA__reg_param': 9.999999999999999e-05}

In [7]:
QDA_search.best_score_

0.6271432896385213

In [8]:
QDA_reg_param_ = QDA_search.best_params_['QDA__reg_param']
qda_model = QuadraticDiscriminantAnalysis(priors = (0.5, 0.5),
                                          reg_param = QDA_reg_param_)
qda_pipeline = Pipeline([('scaler', StandardScaler()), ('QDA', qda_model)])

cross_val_results = pd.DataFrame(cross_validate(qda_pipeline, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["QDA", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
QDA,0.627143,0.701966,0.665863,0.637054


## Logistic Regression

In [9]:
logreg_pipeline = Pipeline([('scaler', StandardScaler()), ('logreg', LogisticRegression(class_weight = "balanced"))])

n = 100
m = 10
Cs = np.logspace(start = -4, stop = 2, num = n)

logreg_search = GridSearchCV(estimator = logreg_pipeline,
                             param_grid = {'logreg__C' : Cs},
                             scoring = 'f1_macro',
                             cv = 5)
logreg_search.fit(X, y)
logreg_search.best_params_

{'logreg__C': 0.6579332246575682}

In [10]:
logreg_search.best_score_

0.6635322595910054

In [11]:
logreg_C = logreg_search.best_params_["logreg__C"]

logreg_model_best = LogisticRegression(C = logreg_C,
                                       class_weight = "balanced")
logreg_pipeline = Pipeline([('scaler', StandardScaler()), ('logreg', logreg_model_best)])

cross_val_results = pd.DataFrame(cross_validate(logreg_pipeline, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["LogReg-Best", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LogReg-Best,0.663532,0.724197,0.681851,0.679451
QDA,0.627143,0.701966,0.665863,0.637054


## Gaussian kernel SVC

In [12]:
rbfsvc = SVC(kernel = "rbf", class_weight = "balanced")
rbfsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", rbfsvc)])

n = 50
m = 50
Cs = np.logspace(start = -1, stop = 2, num = n)
gammas = np.logspace(start = -1, stop = 1, num = m) / X.shape[0]

rbfsvc_search = GridSearchCV(estimator = rbfsvc_pipeline,
                             param_grid = {'svc__C' : Cs,
                                           'svc__gamma' : gammas},
                             scoring = 'f1_macro',
                             cv = 5)
rbfsvc_search.fit(X, y)
rbfsvc_search.best_params_

{'svc__C': 1.4563484775012436, 'svc__gamma': 0.0002687734166234585}

In [13]:
rbfsvc_search.best_score_

0.660344012069942

In [14]:
rbfsvc_C = rbfsvc_search.best_params_['svc__C']
rbfsvc_gamma = rbfsvc_search.best_params_['svc__gamma']
rbfsvc_best = SVC(kernel = "rbf", C = rbfsvc_C, gamma = rbfsvc_gamma, class_weight = "balanced")
rbfsvc_pipeline = Pipeline([("scaler", StandardScaler()), ("svc", rbfsvc_best)])

cross_val_results = pd.DataFrame(cross_validate(rbfsvc_pipeline, X, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["Gaussian SVC", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
LogReg-Best,0.663532,0.724197,0.681851,0.679451
Gaussian SVC,0.660344,0.702813,0.667546,0.68392
QDA,0.627143,0.701966,0.665863,0.637054


## Random Forest

# Voting classifiers

The classifiers, by themselves, have similar performance. However, when predicting the training data they show to have different opinions. A consensus should be established.

In [15]:
qda_pipeline.fit(X, y)
qda_labels = qda_pipeline.predict(X)
logreg_pipeline.fit(X, y)
logreg_labels = logreg_pipeline.predict(X)
rbfsvc_pipeline.fit(X, y)
rbfsvc_labels = rbfsvc_pipeline.predict(X)

In [16]:
pd.Series(qda_labels == logreg_labels).value_counts()

True     396
False     53
Name: count, dtype: int64

In [17]:
pd.Series(qda_labels == rbfsvc_labels).value_counts()

True     398
False     51
Name: count, dtype: int64

In [18]:
pd.Series(rbfsvc_labels == logreg_labels).value_counts()

True     383
False     66
Name: count, dtype: int64

In [19]:
pd.Series(np.logical_and(qda_labels == logreg_labels, qda_labels == rbfsvc_labels)).value_counts()

True     364
False     85
Name: count, dtype: int64

Let's make them vote, then.

In [20]:
estimators = [("logreg", logreg_model_best), ("qda", qda_model), ("rbfsvc", rbfsvc_best)]
votingclass = VotingClassifier(estimators = estimators)
vote_pipeline = Pipeline([("scaler", StandardScaler()), ("voting", votingclass)])

n = 10
weights = [(p1 / n, p2 / n, 1 - p1/n - p2/n) for p1 in range(1, n - 1) for p2 in range(1, n - p1)]
modes = ["hard"]

m1 = 15
m2 = 15
m3 = 15
Cs = np.logspace(start = -1, stop = 2, num = m1)
gammas = np.logspace(start = -1, stop = 1, num = m2) / X.shape[0]
regs = np.logspace(start = -4, stop = -0.5, num = m3)

vote_search = GridSearchCV(estimator = vote_pipeline,
                           param_grid = {"voting__weights" : weights,
                                         "voting__voting" : modes,
                                         "voting__logreg__C" : Cs,
                                         "voting__rbfsvc__C" : Cs,
                                         "voting__rbfsvc__gamma" : gammas,
                                         "voting__qda__reg_param" : regs},
                           scoring = "f1_macro",
                           cv = 5)
vote_search.fit(X, y)
vote_search.best_params_

KeyboardInterrupt: 

In [ ]:
vote_search.best_score_

# AdaBoost